# Lakebase Tables and Sample Records

Evidence that the support app's data lives in **Lakebase** (Databricks-managed Postgres):
the two tables, the foreign key between them, and every record they hold.

Built to be screenshotted — each cell below is one self-contained image.
Run all, then capture cells 2, 3 and 4.

> Connects with `psycopg2`, not `%sql`. A `%sql` cell executes against Unity Catalog,
> which is a different engine and cannot see these tables at all.

In [0]:
import warnings

import psycopg2
import pandas as pd

# pandas warns on every read_sql with a raw DBAPI connection. It works fine with
# psycopg2, and pulling in SQLAlchemy purely to silence it is not worth it.
warnings.filterwarnings("ignore", message=".*only supports SQLAlchemy connectable.*")

try:
    LAKEBASE_URL = dbutils.secrets.get(scope="support", key="lakebase-url")
except NameError:                      # running outside Databricks
    import os
    LAKEBASE_URL = os.environ["LAKEBASE_URL"]

try:
    conn = psycopg2.connect(LAKEBASE_URL)
except psycopg2.Error:                 # password may be percent-encoded
    from urllib.parse import urlparse, unquote
    p = urlparse(LAKEBASE_URL)
    conn = psycopg2.connect(host=p.hostname, port=p.port or 5432,
                            dbname=p.path.lstrip("/"), user=unquote(p.username),
                            password=unquote(p.password), sslmode="require")

try:
    display
except NameError:
    display = lambda df: print(df.to_string(index=False))

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

info = q("SELECT current_user AS postgres_role, current_database() AS database, "
         "inet_server_addr()::text AS server")
print("Connected to Lakebase Postgres")
display(info)

---
## 1. The two tables, and how they relate

In [0]:
print("=== Tables in Lakebase (schema public) ===")
display(q("""
    SELECT 'tickets'         AS table_name, COUNT(*) AS row_count FROM tickets
    UNION ALL
    SELECT 'ticket_messages', COUNT(*)                            FROM ticket_messages
    ORDER BY table_name
"""))

print("=== Columns ===")
display(q("""
    SELECT table_name, ordinal_position AS pos, column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name IN ('tickets', 'ticket_messages')
    ORDER BY table_name, ordinal_position
"""))

print("=== Foreign key: ticket_messages.ticket_id must reference a real ticket ===")
display(q("""
    SELECT tc.table_name        AS from_table,
           kcu.column_name      AS from_column,
           ccu.table_name       AS references_table,
           ccu.column_name      AS references_column,
           rc.delete_rule,
           tc.constraint_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu     ON kcu.constraint_name = tc.constraint_name
    JOIN information_schema.constraint_column_usage ccu ON ccu.constraint_name = tc.constraint_name
    JOIN information_schema.referential_constraints rc  ON rc.constraint_name = tc.constraint_name
    WHERE tc.constraint_type = 'FOREIGN KEY' AND tc.table_name = 'ticket_messages'
"""))

---
## 2. Sample records — `tickets`

Every ticket, with how many messages hang off it.

In [0]:
display(q("""
    SELECT t.ticket_id, t.title, t.status, t.priority, t.category,
           t.created_by, t.created_at,
           COUNT(m.message_id) AS message_count
    FROM tickets t
    LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id
    ORDER BY t.ticket_id
"""))

---
## 3. Sample records — `ticket_messages`

Every message, grouped under the ticket it belongs to. The `ticket_id` column is the
foreign key shown above.

In [0]:
display(q("""
    SELECT m.message_id, m.ticket_id, t.title AS ticket_title,
           m.author, m.created_at, m.message_text
    FROM ticket_messages m
    JOIN tickets t ON t.ticket_id = m.ticket_id
    ORDER BY m.ticket_id, m.created_at, m.message_id
"""))

---
## 4. Requirements met

In [0]:
rows = q("""
    SELECT t.ticket_id, t.status, COUNT(m.message_id) AS messages
    FROM tickets t LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id, t.status ORDER BY t.ticket_id
""")

short    = list(rows[rows.messages < 2].ticket_id)
statuses = sorted(rows.status.unique())

checks = [
    ("two related tables (tickets, ticket_messages)", True),
    ("ticket_messages.ticket_id references tickets",
     not q("""SELECT 1 FROM information_schema.table_constraints
              WHERE constraint_type='FOREIGN KEY' AND table_name='ticket_messages'""").empty),
    (f"at least 3 tickets (have {len(rows)})",             len(rows) >= 3),
    (f"at least 2 statuses (have {len(statuses)}: {', '.join(statuses)})", len(statuses) >= 2),
    (f"at least 2 messages per ticket"
     + ("" if not short else f" - short: {short}"),        not short),
]

display(pd.DataFrame([{"requirement": label, "result": "PASS" if ok else "FAIL"}
                      for label, ok in checks]))
print("\nALL DATA REQUIREMENTS MET" if all(ok for _, ok in checks)
      else "NOT MET: " + "; ".join(l for l, ok in checks if not ok))

conn.close()